# 04 - Michel Electron Mini-Analysis

Goal: build a detector-agnostic Michel electron candidate table from reconstructed SPINE particles, estimate simple selection metrics when truth is available, and send interesting entries to Spinal Tap.

Notebook 2 introduced the YAML config and the object-inspection workflow. Notebook 3 introduced truth matching and validation diagnostics. Here we turn those ingredients into a concrete analysis table.

This notebook builds an analysis table step by step, starting from small object-level questions and ending with a compact selection.

## Physics story in one paragraph

A Michel electron is an electron from a stopped muon decay. In reconstructed SPINE objects, the key ingredients are:

1. a particle with Michel semantic shape;
2. a track-like particle in the same interaction;
3. small spatial separation between the Michel points and the candidate parent track;
4. enough reconstructed points to avoid tiny fragments.

This is not a final detector-specific Michel selection. It is a compact analysis skeleton that works across detectors as long as the HDF5 file contains reconstructed particles.

## Step 1: import the tools for this notebook

This first code cell does not touch any data yet. It only imports the Python modules we will use later.

If an import fails, stop here. There is no point debugging later cells until this environment cell runs cleanly.

In [ ]:

# Standard Python/path tools for working with local files.
from pathlib import Path

# Analysis utilities used throughout the tutorials.
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

# The Driver is the main SPINE entry point for reading one event at a time.
from spine.driver import Driver


## Step 2: choose the input file

This second code cell answers one question: which SPINE HDF5 file should the rest of the notebook read?

It looks for the shared workshop area first, then for the repository-local tutorial assets. It also defines the detector name and sample tag that control which file gets opened.

In [ ]:

# Prefer the shared DUNE path used in the workshop environment.
# Fall back to the repository-local tutorial assets when needed.
# We include both relative spellings because notebook kernels do not always
# start in the notebook directory.
DATA_ROOT_CANDIDATES = [
    Path("/exp/dune/data/users/drielsma/npc-ddas"),
    Path("../assets"),
    Path("tutorials/assets"),
]

# Pick the first location that actually exists on disk.
DATA_ROOT = next((path for path in DATA_ROOT_CANDIDATES if path.exists()), None)
if DATA_ROOT is None:
    raise RuntimeError("Could not find a workshop data directory.")

# The reco HDF5 files live under reco/ inside the chosen data root.
HDF5_DATA_DIR = DATA_ROOT / "reco"

# Edit these values to switch samples.
# The expected layout is:
#   reco/DETECTOR/SAMPLE_NAME_spine.h5
DETECTOR = "2x2"
SAMPLE_NAME = "2x2_numi"

# GEOMETRY tells SPINE which detector geometry description to use.
# For this workshop we use the detector name directly.
GEOMETRY = DETECTOR
HDF5_FILE_NAME = f"{SAMPLE_NAME}_spine.h5"

# Build the final path to the reconstructed SPINE HDF5 file.
DATA_PATH = HDF5_DATA_DIR / DETECTOR / HDF5_FILE_NAME

print(f"Using data root: {DATA_ROOT}")
print(f"Reco file: {DATA_PATH}")


## Step 3: build the SPINE driver from YAML

Now that the notebook knows which file to read, it loads the tutorial YAML config, injects the concrete file path, and creates a `Driver` object.

The geometry override in this step is important: it tells SPINE which detector geometry description to attach when the chosen sample needs one.

In [ ]:

# As with the data paths, the notebook working directory can vary.
# These two candidates point to the same tutorial config from different cwd values.
CONFIG_CANDIDATES = [
    Path("../config/read_spine_hdf5.yaml"),
    Path("tutorials/config/read_spine_hdf5.yaml"),
]
CONFIG_PATH = next((path for path in CONFIG_CANDIDATES if path.exists()), None)

# Replace the DATA_PATH placeholder in the YAML template with the file we chose above.
DATA_PATH = str(DATA_PATH)
if CONFIG_PATH is None:
    raise RuntimeError("Could not find tutorials/config/read_spine_hdf5.yaml")

# Read the YAML as text first so we can substitute the concrete file path.
cfg_text = CONFIG_PATH.read_text().replace("DATA_PATH", DATA_PATH)

# Convert the YAML text into a normal Python dictionary.
cfg = yaml.safe_load(cfg_text)

# Some detector samples need an explicit geometry block so downstream code knows
# which detector layout and coordinate conventions to use.
if GEOMETRY:
    cfg["geo"] = {"detector": GEOMETRY}

# Create the SPINE driver. From this point on, driver.process(entry=...) reads events.
driver = Driver(cfg)
print(f"Opened {DATA_PATH}")
print(f"Entries: {len(driver)}")
if GEOMETRY:
    print(f"Detector geometry: {GEOMETRY}")


## Reuse the setup from Notebook 2

    This notebook uses the same readback config as Notebook 2. The full YAML appears there.

The new work in this notebook is the analysis logic, not the file-loading logic.

In [ ]:
N_ENTRIES = min(len(driver), 50)
print(f"Scanning {N_ENTRIES} entries")

In [ ]:
from spine.constants import MICHL_SHP, TRACK_SHP, SHAPE_LABELS

print(f"Michel shape id: {MICHL_SHP} -> {SHAPE_LABELS[MICHL_SHP]}")
print(f"Track shape id:  {TRACK_SHP} -> {SHAPE_LABELS[TRACK_SHP]}")

## Start with one event, not fifty

Before writing a batch loop, inspect one event and ask simpler questions:

1. Does this event contain any Michel-shaped particles?
2. If yes, which interaction do they belong to?
3. Is there a track-like particle in that same interaction?

In [ ]:
EXAMPLE_ENTRY = 0
example = driver.process(entry=EXAMPLE_ENTRY)
example_particles = example.get("reco_particles", [])

In [ ]:
example_particle_df = pd.DataFrame(
    [
        {
            "id": getattr(p, "id", -1),
            "interaction_id": getattr(p, "interaction_id", -1),
            "shape": getattr(p, "shape", -1),
            "shape_label": SHAPE_LABELS.get(getattr(p, "shape", -1), "unknown"),
            "size": getattr(p, "size", np.nan),
            "depositions_sum": getattr(p, "depositions_sum", np.nan),
        }
        for p in example_particles
    ]
)

example_particle_df.head(20)

In [ ]:
example_michels = example_particle_df.query("shape == @MICHL_SHP")
example_michels

## Pause on the distance function

The line `diff = a[:, None, :] - b[None, :, :]` builds all pairwise point differences between two particles. That gives us the closest approach between a Michel candidate and a track without relying on detector-specific geometry.

If that NumPy expression feels unfamiliar, that is fine. Treat it as a utility function and focus on what goes in and what comes out.

In [ ]:
def points(p):
    pts = getattr(p, "points", None)
    if pts is None:
        return np.empty((0, 3))
    return np.asarray(pts)


def min_point_distance(p1, p2):
    a = points(p1)
    b = points(p2)
    if len(a) == 0 or len(b) == 0:
        return np.inf
    diff = a[:, None, :] - b[None, :, :]
    return float(np.sqrt(np.sum(diff * diff, axis=-1)).min())


def deposition_sum(p):
    value = getattr(p, "depositions_sum", None)
    if value is not None:
        return float(value)
    depositions = getattr(p, "depositions", None)
    return float(np.sum(depositions)) if depositions is not None else np.nan

## Choose simple thresholds

These numbers are analysis choices, not universal truths. The point of the notebook is to make those choices visible and easy to change.

In [ ]:
MUON_MIN_POINTS = 20
ATTACH_THRESHOLD_CM = 3.0
MICHEL_MIN_POINTS = 5
MATCH_THRESHOLD = 0.1

print({
    "muon_min_points": MUON_MIN_POINTS,
    "attach_threshold_cm": ATTACH_THRESHOLD_CM,
    "michel_min_points": MICHEL_MIN_POINTS,
    "match_threshold": MATCH_THRESHOLD,
})

In [ ]:
def best_attached_track(candidate, particles):
    same_interaction_tracks = [
        p for p in particles
        if getattr(p, "interaction_id", -1) == getattr(candidate, "interaction_id", -2)
        and getattr(p, "shape", -1) == TRACK_SHP
        and getattr(p, "size", 0) >= MUON_MIN_POINTS
    ]
    if not same_interaction_tracks:
        return None, np.inf
    distances = [(track, min_point_distance(candidate, track)) for track in same_interaction_tracks]
    return min(distances, key=lambda item: item[1])


def best_truth_match(p):
    ids = list(getattr(p, "match_ids", []))
    overlaps = list(getattr(p, "match_overlaps", []))
    if not ids or not overlaps:
        return -1, 0.0
    index = int(np.argmax(overlaps))
    return ids[index], float(overlaps[index])


def michel_candidate_row(entry, p, particles):
    parent_track, attach_dist = best_attached_track(p, particles)
    match_id, match_overlap = best_truth_match(p)
    return {
        "entry": entry,
        "particle_id": getattr(p, "id", -1),
        "interaction_id": getattr(p, "interaction_id", -1),
        "size": getattr(p, "size", np.nan),
        "depositions_sum": deposition_sum(p),
        "is_contained": getattr(p, "is_contained", None),
        "attach_dist_cm": attach_dist,
        "attached_track_id": getattr(parent_track, "id", -1) if parent_track is not None else -1,
        "attached_track_size": getattr(parent_track, "size", np.nan) if parent_track is not None else np.nan,
        "match_id": match_id,
        "match_overlap": match_overlap,
        "is_truth_matched": match_overlap >= MATCH_THRESHOLD,
        "ke": getattr(p, "ke", np.nan),
    }

## Build the candidate table

Pause on `p.shape == MICHL_SHP`: this is where the high-level analysis becomes a SPINE-object query. Everything after that is ordinary table building.

Read the loop with this question in mind: which lines are SPINE-specific, and which lines are just normal Python and pandas?

In [ ]:
rows = []
true_rows = []

for entry in range(N_ENTRIES):
    data = driver.process(entry=entry)
    reco_particles = data.get("reco_particles", [])
    truth_particles = data.get("truth_particles", [])

    # First collect reconstructed Michel candidates.
    for p in reco_particles:
        if getattr(p, "shape", -1) == MICHL_SHP:
            rows.append(michel_candidate_row(entry, p, reco_particles))

    # Then collect truth Michels so we can estimate rough efficiency.
    for tp in truth_particles:
        if getattr(tp, "shape", -1) == MICHL_SHP:
            match_id, match_overlap = best_truth_match(tp)
            true_rows.append({
                "entry": entry,
                "truth_particle_id": getattr(tp, "id", -1),
                "interaction_id": getattr(tp, "interaction_id", -1),
                "size": getattr(tp, "size", np.nan),
                "depositions_sum": deposition_sum(tp),
                "is_contained": getattr(tp, "is_contained", None),
                "match_id": match_id,
                "match_overlap": match_overlap,
                "is_reco_matched": match_overlap >= MATCH_THRESHOLD,
                "ke": getattr(tp, "ke", np.nan),
            })

In [ ]:
candidate_columns = [
    "entry",
    "particle_id",
    "interaction_id",
    "size",
    "depositions_sum",
    "is_contained",
    "attach_dist_cm",
    "attached_track_id",
    "attached_track_size",
    "match_id",
    "match_overlap",
    "is_truth_matched",
    "ke",
]
truth_columns = [
    "entry",
    "truth_particle_id",
    "interaction_id",
    "size",
    "depositions_sum",
    "is_contained",
    "match_id",
    "match_overlap",
    "is_reco_matched",
    "ke",
]
candidates = pd.DataFrame(rows, columns=candidate_columns)
truth_michels = pd.DataFrame(true_rows, columns=truth_columns)

display(candidates.head())
display(truth_michels.head())

## Micro-exercise: interrogate one candidate row

Pick one row from `candidates` and answer:

1. What interaction does it belong to?
2. What track did we attach it to?
3. Is the attachment distance small or large?
4. If truth is available, does the best overlap look convincing?

In [ ]:
if len(candidates):
    candidates["passes_michel_selection"] = (
        (candidates["size"] >= MICHEL_MIN_POINTS)
        & (candidates["attach_dist_cm"] <= ATTACH_THRESHOLD_CM)
    )
else:
    candidates["passes_michel_selection"] = []

summary = candidates.groupby("passes_michel_selection").size().rename("n_reco_michel_candidates").to_frame()
summary

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3))
if len(candidates):
    candidates["size"].hist(ax=axes[0], bins=30)
    candidates["attach_dist_cm"].replace(np.inf, np.nan).hist(ax=axes[1], bins=30)
    candidates["depositions_sum"].hist(ax=axes[2], bins=30)
axes[0].set_xlabel("Michel candidate points")
axes[1].set_xlabel("closest track distance [cm]")
axes[2].set_xlabel("candidate deposition sum")
plt.tight_layout()

In [ ]:
selected = candidates[candidates["passes_michel_selection"]].copy()
selected.head(20)

## Truth metrics, if truth is available

These are intentionally simple counts. They are meant to start a discussion, not to certify a production performance number.

In [ ]:
n_pred = int(candidates["passes_michel_selection"].sum()) if len(candidates) else 0
n_pred_matched = int((candidates["passes_michel_selection"] & candidates["is_truth_matched"]).sum()) if len(candidates) else 0
n_true = len(truth_michels)
n_true_matched = int(truth_michels["is_reco_matched"].sum()) if len(truth_michels) else 0

purity = n_pred_matched / n_pred if n_pred else np.nan
efficiency = n_true_matched / n_true if n_true else np.nan

pd.DataFrame([{
    "selected_reco_michels": n_pred,
    "selected_truth_matched": n_pred_matched,
    "true_michels": n_true,
    "true_reco_matched": n_true_matched,
    "rough_purity": purity,
    "rough_efficiency": efficiency,
}])

## SPINE-native CSV export

The derived Michel table above is useful for teaching and for quick iteration. But when you want to dump raw SPINE object information to CSV, use SPINE's save analysis script instead of hand-writing lots of export code.

This is a good division of labor:

- use normal notebook code for derived quantities such as `attach_dist_cm` or custom selections;
- use the save script for standard object attributes that already exist on particles and interactions.

In [ ]:
# This config is a compact example of what you could put in a standalone YAML file.
save_cfg = {
    "save": {
        "obj_type": ["particle", "interaction"],
        "run_mode": "both",
        "match_mode": "both",
        "overwrite": True,
        "particle": [
            "id",
            "interaction_id",
            "pid",
            "shape",
            "size",
            "depositions_sum",
            "is_primary",
        ],
        "interaction": [
            "id",
            "nu_id",
            "size",
            "topology",
            "vertex",
        ],
    }
}

print(yaml.safe_dump({"ana": save_cfg}, sort_keys=False))

In [ ]:
from spine.ana.manager import AnaManager

SAVE_OUTPUT_DIR = Path("save_demo_output")
SAVE_OUTPUT_DIR.mkdir(exist_ok=True)

save_manager = AnaManager(save_cfg, log_dir=str(SAVE_OUTPUT_DIR))
# Run the save script on a small sample first so the output stays easy to inspect.
for entry in range(min(10, len(driver))):
    save_manager(driver.process(entry=entry))
save_manager.close()

sorted(path.name for path in SAVE_OUTPUT_DIR.glob("save_*.csv"))

In [ ]:
saved_particles = pd.read_csv(SAVE_OUTPUT_DIR / "save_reco_particles.csv")
saved_particles.head()

## Spinal Tap handoff

The useful debugging output is a small list of selected candidates and suspicious failures:

- selected Michel candidates with large attachment distance;
- selected candidates with no truth match;
- true Michels with no reco match;
- high-charge candidates that fail the attachment cut.

This CSV is still custom because it stores derived quantities from our notebook selection logic, not just raw SPINE object attributes.

In [ ]:
event_list = candidates.sort_values(
    ["passes_michel_selection", "is_truth_matched", "entry", "particle_id"],
    ascending=[False, True, True, True],
)
event_list[[
    "entry",
    "particle_id",
    "interaction_id",
    "passes_michel_selection",
    "is_truth_matched",
    "size",
    "attach_dist_cm",
    "attached_track_id",
    "depositions_sum",
]].to_csv("spinal_tap_michel_event_list.csv", index=False)
print("Wrote spinal_tap_michel_event_list.csv")
event_list.head(20)

## Live exercise

Choose one:

1. Change `ATTACH_THRESHOLD_CM` from 3 cm to 1 cm or 5 cm. What happens to the candidate count?
2. Raise `MICHEL_MIN_POINTS`. Which candidates disappear first?
3. Open one unmatched selected candidate in Spinal Tap. Is it a real Michel, a delta ray, a fragment, or a shower piece?
4. Open one true unmatched Michel in Spinal Tap. Was it missed by segmentation, clustering, or matching?
5. Compare one row from `saved_particles` with one row from `candidates`. Which columns came directly from SPINE, and which ones did we derive ourselves?

## Offline extensions

- Add a parent-muon stopping requirement using the track endpoint and nearby charge.
- Estimate a rough Michel energy spectrum from `depositions_sum` or `ke`, then compare matched reco and truth.
- Split efficiency by candidate size, containment, or detector region.
- Build a curated Spinal Tap list of five clean Michels and five failure modes.
- Try the same notebook on a second detector sample and identify which thresholds need retuning.